***Sound Event Detection***  
This notebook uses the panns_inference and webrtcvad libraries to explore the various sonic elements of a peice of audio. It particularly focuses on the detection and location of human speech within audio. This notebook was designed to assist exploration of audio recordings from the PARADISEC collection.

This code loads in the .wav file which is to be analyzed, then does the majority of the data processing. It ultimately creates the audio_tags and speech_tags lists, which contain audio labels for each second of the audio.

In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
from panns_inference import AudioTagging, SoundEventDetection, labels
#Add the audio path to the clip you want to analyze
audio_file_path=""
#Loads the audio as a Librosa object, and also obtains the sample rate of the clip
(audio,sr)=librosa.core.load(audio_file_path,sr=48000,mono=True)
#Graphs the spectrogram representation of the audio using matplotlib
S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128,fmax=8000)
fig, ax = plt.subplots()
S_dB = librosa.power_to_db(S, ref=np.max)
img = librosa.display.specshow(S_dB, x_axis='time',y_axis='mel', sr=sr,fmax=8000, ax=ax)
fig.colorbar(img, ax=ax, format='%+2.0f dB')
ax.set(title='Mel-frequency spectrogram')

In [ ]:
import librosa
import panns_inference
import numpy as np
import webrtcvad
from pydub import AudioSegment
#Sets the length of each audio chunk in seconds (the length of audio clips which will be labelled)
chunk_duration=1
#The number of audio samples in each chunk
chunk_samples=int(sr*chunk_duration)
#Creates a list of all the audio chunks to be analyzed
chunks = [audio[i:i + chunk_samples] for i in range(0, len(audio)-len(audio)%chunk_samples, chunk_samples)]
#Prepares the lists of audio tags
audio_tags=[]
speech_tags=[]
#Creates audio tagging and speech recognition objects
at=AudioTagging(checkpoint_path=None, device='cuda')
vad=webrtcvad.Vad()
#Iterates through every audio chunk
for i in chunks:
  #Converts the chunk into a form which can be fed into the tagging/recognition objects
  inp16 = (i * 32767).astype(np.int16)
  inp = i[None, :]
  num_splits=100
  rt=0
  for j in range(num_splits):
    #Splits the audio chunk into further smaller pieces for the speech recognition to analyze (100 smaller sub-clips)
    temp=inp16[j*sr*chunk_duration//num_splits:(j+1)*sr*chunk_duration//num_splits]
    #Passes each sub-clip to the speech recognition object (webrtcVAD)
    if vad.is_speech(temp.tobytes(),sr):
      rt+=1
    #If more than half (an arbitrary choice of value, can fine tune if a more permissive or restrictive model is desired) of the sub-clips return as having speech, label the chunk as containing speech
  if rt>=50:
    speech_tags.append(True)
  else:
    speech_tags.append(False)
  #Pass the chunks into the audio tagging object (PANNs inference)
  chunkwise_output,embedding=at.inference(inp)
  #Extract the useful information (confidence scores for the tags) from the results
  scores = chunkwise_output[0]
  #Find and save the six highest confidence labels for each chunk
  top_indices = np.argsort(scores)[::-1][:6]
  audio_tags.append([])
  for j in top_indices:
    #Will be in the form of ("Speech",0.63)
    audio_tags[-1].append((labels[j],scores[j]))

This code returns the audio labels and speech detection for each requested second of the audio. Enter a range of seconds (ie. 0-5, 52-60) to see the information for each.

In [ ]:
high=0
low=0
#Ensures the user inputs a valid range
while not (low<high):
  #Accepts the range of seconds the user wants information for
  indices=input("Which range of seconds would you like information for? (ie. 0-10, 20-22): ")
  low,high=indices.split("-")
  low=int(low)
  high=int(high)
  #Ensures the user inputs a valid range
  if not (low<high):
    print("Not a valid range")
#Goes through the requested range
for i in range(low,high+1):
  print("Second "+str(i)+": ")
  #Prints the audio tags (PANNs inference)
  for j in audio_tags[i]:
    print(j[0]+": "+str(j[1])) #Prints the speech tags (webrtcVAD)
  print("VAD speech detected: "+str(speech_tags[i]))
  print()


This code uses the previously created tags to create a "smart detection" of speech in the audio file. By combining the speech data recieved from the VAD and SED, it predicts whether each second of audio contains speech. By combining the data from two sources, this method provides a more accurate prediction then either source individually.

In [ ]:
speech=[]
#Goes through every second of the full clip and assigns either a 0 (no speech), 1 (possibly speech), or 2 (probably speech) to each second
for i in range(len(audio_tags)):
  speech.append(0)
  rs=0
  #If one of the "speech-like" tags (from PANNs inference) is the highest confidence tag, or has a confidence greater than 0.6, assigns the second as a 2
  for j in range(len(audio_tags[i])):
    if audio_tags[i][j][0] in ["Speech","Narration, monologue","Male speech, man speaking","Female speech, woman speaking","Mantra"]:
      if audio_tags[i][j][1]>0.6 or j==0:
        speech[-1]=2
        break
      #If a speech-like tag is the second most confident tag above a confidence of 0.1, assigns the second as a 1
      if j==1 and audio_tags[i][0][1]<0.4 and audio_tags[i][j][1]>.1:
        speech[-1]=1
      #Sums all the speech-like tags
      rs+=audio_tags[i][j][1]
  #If the sum of the confidences of all speech-like tags is greater than or equal to 1.2, assigns the second as a 2
  if rs>=1.2:
    speech[-1]=2
  #If the speech tag (from webrtcVAD) is true, assigns the second as a 2
  elif speech_tags[i]:
    speech[-1]=2
    #If the sum of all speech-like tags is greater than or equal to 0.8 (but not greater than or equal to 1.2, due to the above condition) assigns the second as a 1
  elif rs>=.8 and speech[-1]==0:
    speech[-1]=1
final=[]
#Goes through every second of the clip and, based on its context, classifies every second of audio as definitively speech or no speech
#If the 1st second is a 1 and the 2nd second is a 2, assigns the first second as having speech, otherwise, no speech. If the 1st second is a 2, then assigns it as speech. If it is a 0, then no speech.
if (speech[0]==1 and speech[1]==2) or speech[0]==2:
  final.append(True)
else:
  final.append(False)
#For seconds 2 through the penultimate, assigns every 2 as being speech, every 0 as being no speech, and a 1 as speech only if it is surrounded by either two 2s or a 1 and a 2, otherwise no speech.
for i in range(1,len(speech)-1):
  if speech[i]==2:
    final.append(True)
  elif speech[i]==1 and (speech[i-1]+speech[i+1]>=3):
    final.append(True)
  else:
    final.append(False)
  #If the last seond is a 2, it is assigned as speech. If it is a 0, no speech. If the last second is a 1 and the penultimate second is a 2, assigns the last second as speech, otherwise no speech.
if speech[-1]==2 or (speech[-1]==1 and speech[-2]==2):
  final.append(True)
else:
  final.append(False)
#Prints a list of the final results (True/False based on whether speech is present)
for i in range(len(final)):
  print(str(i)+" "+str(final[i]))

#This code was initially done with a single-pass method (going directly from the audio/speech tags to a final result), but resulted in a large number of false positives/negatives. By first locating seconds which were not decisively speech or no speech and considering their context before finalizing a label, many of the false positives/negatives were eliminated.




This code does further label prediction on the audio, structuring the resulting tags in a way that allows the tags to be graphed over time.

 The SoundEventDetection object from PANNs Inference uses fewer neural networks to make decisions, making it less accurate but capable of creating higher-sample-rate predictions with less computation (resulting in a smoother graph) when compared to AudioTagging (also from PANNs Inference). I chose AudioTagging over SoundEventDetection for the main portion of this notebook, but use SoundEventDetection here.

In [ ]:
import matplotlib.pyplot as plt
#Creates a SoundEventDetection object (PANNs Inference)
sed = SoundEventDetection(checkpoint_path=None, device='cuda')
#Formats the audio in a way that can be used by the SoundEventDetection object
audioin=audio[None, :]
#Passes the audio to the SED object, and saves the resulting output
framewise_output = sed.inference(audioin)

Graphs the confidence of the five most prevalent audio tags over the course of the audio. Saves the graph as "output_graph.png"

In [ ]:
#Extracts the audio tags from the SED output (see previous code cell)
output = framewise_output[0]
top_n = 5
#Sizes the graph based on the length of the audio clip (the wide range of possible audio clip lengths makes a fixed size graph useless in most cases)
time = np.arange(output.shape[0])/48000*320
plt.figure(figsize=(max(round(max(time)/10),7), 5))
#Determines the five most prevalent audio tags across the clip
top_classes = np.argsort(output.max(axis=0))[::-1][:top_n]
#Plots the confidence of the audio tags over time
for i in top_classes:
    label = sed.labels[i]
    plt.plot(time, output[:, i], label=label)
plt.legend()
plt.title("Audio Tag Confidence Over Time")
plt.xlabel("Time (s)")
plt.ylabel("Label Confidence")
plt.savefig("output_graph.png")